In [1]:
import warnings
import os
import sys
import pandas as pd
warnings.filterwarnings("ignore")
username = os.getlogin()
sys.path.append(rf'/home/{username}/data_share_p')
import FactorFramework.FactorFramework as ff
from FactorFramework.FactorFramework.global_config import label_config_dict

In [2]:
result_save_path_root = rf'/home/{username}/data_share/StockOutputData'

In [3]:
# 获取全部因子的因子信息（表达式）
output_type_list = [x for x in os.listdir(result_save_path_root) if '_' not in x]
fac_exp = dict()
for output_type in output_type_list:
    # 遍历每个因子系列
    for fac_name_ser in os.listdir(rf'{result_save_path_root}/{output_type}'):
        # 遍历系列下的所有因子
        for fac_name in os.listdir(rf'{result_save_path_root}/{output_type}/{fac_name_ser}'):
            # 因子信息
            fac_info = pd.read_pickle(rf'{result_save_path_root}/{output_type}/{fac_name_ser}/{fac_name}/fac_info.pkl')
            fac_exp[fac_name] = fac_info['fac_exp']
fac_exp = pd.Series(fac_exp).to_frame().reset_index()
fac_exp.columns = ['fac_name', 'fac_exp']

In [4]:
# 获取全部因子的回测信息
output_label_type_list = [x for x in os.listdir(result_save_path_root) if '_'  in x]
fac_backtest_result = []
for output_label_type in output_label_type_list:
    label_name = output_label_type.split('_')[-1]
    valid_mask_name = label_config_dict[label_name].valid_mask_name
    # 遍历每个因子系列
    for fac_name_ser in os.listdir(rf'{result_save_path_root}/{output_label_type}'):
        # 遍历系列下的所有因子
        for fac_name in os.listdir(rf'{result_save_path_root}/{output_label_type}/{fac_name_ser}'):
            # 因子回测表
            fac_info = pd.read_pickle(rf'{result_save_path_root}/{output_label_type}/{fac_name_ser}/{fac_name}/res_data_eval_mean_{valid_mask_name}_{label_name}.pkl')
            fac_info['fac_name'] = fac_name
            fac_info['label_name'] = label_name
            fac_info = fac_info.loc[fac_info['period_name'] == 'all_period_mean']
            fac_backtest_result.append(fac_info)
fac_backtest_result = pd.concat(fac_backtest_result, axis=0, ignore_index=True)

In [5]:
# 合并表达式和回测结果
fac_info = fac_exp.merge(fac_backtest_result, on='fac_name', how='outer')

In [6]:
fac_info_sorted = fac_info[fac_info['label_name'] == 'label5'].sort_values(by='ic', ascending=False)
fac_info_sorted

,fac_name,fac_exp,period_name,ic,rank_ic,group1,group2,group3,group4,group5,...,group20,tail50,tail100,tail200,head50,head100,head200,nan_proportion,zero_proportion,label_name
698,test_2383eb00b1316dc967b28120e9a6538e,"ts_regres(cs_winsorize(bv_skew), div(log(ma_12...",all_period_mean,0.020354,0.037749,0.160815,0.053917,0.174304,0.123879,0.256598,...,0.736556,0.139938,0.151475,0.161453,0.750441,0.724543,0.741724,0.0,0.0,label5
2340,test_20770a9e928dd5cce59e12b68ba20e40,normal_cdf(cs_group_rank_neutralize(neg(ask_su...,all_period_mean,0.019843,0.035986,0.159523,0.060260,0.098801,0.127663,0.228213,...,0.768622,0.071858,0.099024,0.153731,0.917821,0.874496,0.768299,0.0,0.0,label5
1827,test_9c81ffb9a62eefc2001d6a688f0ba06a,"cs_rank_quantile(sub(div(bid_curv, bid_curv_no...",all_period_mean,0.019340,0.037317,0.051055,0.156703,0.166056,0.102182,0.259754,...,0.761039,0.041645,0.102236,0.063571,0.907649,0.857104,0.763549,0.0,0.0,label5
806,test_5c5ddae776bbbaac4c840571d7484eac,"log(normal_cdf(Pow(tp_ma_10, -0.5)))",all_period_mean,0.019191,0.037109,0.163119,0.048081,0.155988,0.110771,0.253497,...,0.767754,0.144992,0.161404,0.172557,0.904595,0.874977,0.763961,0.0,0.0,label5
3928,test_ba9c912efbe0a6f211a82848134b403f,cs_rank_mad(cs_salience(normal_log_prob(bid_su...,all_period_mean,0.019177,0.037234,0.061732,0.118809,0.133616,0.199022,0.285873,...,0.772659,-0.150686,0.007637,0.047620,0.915035,0.872577,0.769281,0.0,0.0,label5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6668,test_408dd5d9da11518f90e7c508dc6ece49,"ts_cov(arctan(ts_regres(ask_submit_price, IV_h...",all_period_mean,NaN,-0.008607,0.484826,0.557743,0.482537,0.383682,0.503089,...,0.096124,0.386044,0.442156,0.482960,0.233839,0.140151,0.101627,0.0,1.0,label5
6680,test_4fac609eb8599ea6099f3ec6e6fe4bb7,"cs_min(ts_cov(sv_skew, ts_regres(bid_ratio, lo...",all_period_mean,NaN,-0.008607,0.484826,0.557743,0.482537,0.383682,0.503089,...,0.096124,0.386044,0.442156,0.482960,0.233839,0.140151,0.101627,0.0,1.0,label5
6683,test_3e0a569efadd31f316b45db3afe35374,"ts_delay(cs_median(log(RVSJP)), 3)",all_period_mean,NaN,-0.008607,NaN,NaN,NaN,NaN,NaN,...,0.447642,NaN,NaN,NaN,0.233839,0.140151,0.101627,1.0,0.0,label5
6686,test_e2a74be4976b5a67659b6ca57a798613,"ts_argmax(ts_obnm(rolling_fft_log_vol_mean, lo...",all_period_mean,NaN,-0.008607,0.484826,0.557743,0.482537,0.383682,0.503089,...,0.096124,0.386044,0.442156,0.482960,0.233839,0.140151,0.101627,0.0,0.0,label5


In [7]:
fac_info_sorted['fac_exp'].iloc[2]

'cs_rank_quantile(sub(div(bid_curv, bid_curv_norm), ma_24))'